# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ishita2004/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Selected Task Type:** Ranking / Scoring (backed by probabilistic binary classification)

**Why this task type:** The business goal is not to output a simple binary decision ("fix this page" vs "ignore it"). Content marketing teams operate under strict capacity constraints: an editorial team might only have bandwidth to review 20 to 50 articles per week. Therefore, the problem is fundamentally a **ranking problem** — ordering the content portfolio by estimated decay risk and traffic opportunity. By assigning a probabilistic opportunity score to each page, we can rank all published content and present editors with a prioritized queue where the top K items deliver the highest expected return on editorial time.

In [1]:
# Task type confirmation
print('ML Task Type: Ranking / Priority Scoring')
print('Output: A continuous score [0, 100] used to rank pages for review capacity K.')


ML Task Type: Ranking / Priority Scoring
Output: A continuous score [0, 100] used to rank pages for review capacity K.


## 2. Target or proxy

- **Target Definition:** `is_declining_label = (trend_direction.str.lower() == 'down').astype(int)`
- **Target Origin:** Derived from *observed search performance outcomes* (traffic trends and impressions over time), NOT from a human rule or product decision flag.
- **Future Target Extension:** In the full warehouse, the target extends to future-window outcome prediction: prior 90-day feature window -> decline over subsequent 30-day window (`impressions_next30d < 0.8 * impressions_prior90d`).
- **Strict Leakage Prevention:** Neither `trend_direction` nor `trend_pct` may ever be used as input features, because they directly encode the target label.

In [2]:
# Verification of Target Definition & Leakage Safeguard
target_name = 'is_declining_label'
prohibited_features = ['trend_direction', 'trend_pct']
print(f'Target Variable: {target_name}')
print(f"Prohibited Input Features (to prevent leakage): {', '.join(prohibited_features)}")


Target Variable: is_declining_label
Prohibited Input Features (to prevent leakage): trend_direction, trend_pct


## 3. Success metric

- **Primary Metric:** **Precision@K (specifically Precision@50)**
- **Why Precision@K:** Standard accuracy is misleading due to class imbalance and because editors only act on the top K recommendations. Precision@50 directly measures real-world utility: *'Of the top 50 pages the system recommends for review, what fraction are genuinely declining?'*
- **Baseline Benchmark:** The hand-written rule baseline achieves Precision@50 = **0.240** (12 out of 50 correct). A successful ML model must achieve Precision@50 >= **0.500** (25+ out of 50 correct) under **client-holdout validation**.

In [3]:
def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

print('Primary Metric Defined: Precision@K (K=50)')
print('Target Benchmark: > 0.500 Precision@50 on out-of-sample client holdout.')


Primary Metric Defined: Precision@K (K=50)
Target Benchmark: > 0.500 Precision@50 on out-of-sample client holdout.


## 4. The unit of analysis, as a real dataframe

**Grain:** One row = **One pseudonymized content item (`content_id`)** for a specific client (`client_id`).

The code cell below loads the starter dataset slice, verifies the grain, constructs the target column, and displays the exact input feature matrix.

In [4]:
import os, sys, pandas as pd, numpy as np

# Ensure kernel is at repo root
while not os.path.isdir('data/raw') and os.getcwd() != os.path.abspath(os.sep):
    os.chdir('..')

# Load starter dataset slice
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# Filter valid candidate pages
df_slice = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].copy()

# Define target column
df_slice['is_declining_label'] = df_slice['trend_direction'].str.lower().eq('down').astype(int)

# Verify Grain: one row per content_id
is_unique_grain = df_slice['content_id'].nunique() == len(df_slice)

print(f'Dataset Slice Shape: {df_slice.shape[0]:,} rows x {df_slice.shape[1]} columns')
print(f'Grain Verification (One row per content_id): {is_unique_grain}')
print(f'Declining Label Rate: {df_slice["is_declining_label"].mean():.3f}')

# Display sample dataframe slice with features and target
feature_cols = ['content_id', 'client_id', 'impressions_90d', 'days_since_last_update', 'avg_position', 'ctr', 'word_count', 'is_declining_label']
df_slice[feature_cols].head(5)


Dataset Slice Shape: 30,000 rows x 45 columns
Grain Verification (One row per content_id): True
Declining Label Rate: 0.542


,content_id,client_id,impressions_90d,days_since_last_update,avg_position,ctr,word_count,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,20,10.6,0.76,3221.0,1
1,content_a1fb4e703a9e,client_4e07408562,15320,25,20.3,0.05,2481.0,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,20,36.5,0.09,3515.0,1
3,content_331d6c4de07b,client_19581e27de,11751,22,6.2,0.49,NaN,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,14,44.0,0.13,2803.0,1


## 5. Why ML beats a fixed rule here

1. **Multi-Signal Non-Linear Interactions:** Fixed rules use rigid, independent cutoffs (e.g. `age >= 180 AND impressions >= 500`). However, a page with 450 impressions at average position 2 with a collapsing CTR is a far higher priority than a page with 550 impressions at position 45 that has been stale for 200 days. Fixed rules miss these trade-offs.
2. **Continuous Scoring vs Block Ties:** A hand-written rule categorizes pages into coarse buckets, creating massive ties at the top of the queue. A machine learning model assigns fine-grained probability scores that rank candidates smoothly.
3. **Adaptability Across Portfolios:** Different clients have varying traffic baselines and position profiles. An ML model learns non-linear feature interactions (such as CTR expectation relative to position tier) that generalize across unseen clients.

In [5]:
# Demonstrating tie fragmentation: Rule vs Continuous Model Score
stale = (df_slice['days_since_last_update'] >= 180).astype(int)
visible = (df_slice['impressions_90d'] >= 500).astype(int)
df_slice['hand_rule'] = stale * visible * df_slice['impressions_90d']

unique_rule_scores = df_slice['hand_rule'].nunique()
print(f'Unique values in Hand Rule Score: {unique_rule_scores}')
print('ML advantage: Continuous non-linear scoring breaks tied heuristic blocks and captures subtle signal interactions.')


Unique values in Hand Rule Score: 18
ML advantage: Continuous non-linear scoring breaks tied heuristic blocks and captures subtle signal interactions.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.